# ChessHack — all-in-one (48 cores + 80GB GPU)

Everything runs on this one machine: **label** on the CPU cores, **distill-train** on the GPU, then **self-play** (CPU workers + GPU inference server). Checkpoints/shards/Elo persist to Google Drive so a session restart loses nothing.

**Before running:** set `REPO_URL` (cell 4) to your GitHub repo. Runtime → Change runtime type → **GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import os; print('CPU cores:', os.cpu_count())

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE='/content/drive/MyDrive/chesshack'
for d in ('data','data/distill','data/nets','bench'): os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
print('persisting to', DRIVE)

In [ ]:
import os
REPO_URL='https://github.com/anilmercanoglu-debug/chesshack.git'   # <-- set this
if not os.path.isdir('/content/chesshack/.git'):
    !git clone $REPO_URL /content/chesshack
else:
    !cd /content/chesshack && git pull
%cd /content/chesshack
!git log --oneline -1

In [ ]:
# deps: torch ships with Colab (CUDA). add python-chess; fetch Stockfish into tools/
!pip -q install python-chess
import os, urllib.request, tarfile, shutil, glob
os.makedirs('tools', exist_ok=True)
if not os.path.exists('tools/stockfish'):
    url='https://github.com/official-stockfish/Stockfish/releases/latest/download/stockfish-ubuntu-x86-64-avx2.tar'
    urllib.request.urlretrieve(url,'/tmp/sf.tar'); tarfile.open('/tmp/sf.tar').extractall('/tmp')
    b=[f for f in glob.glob('/tmp/stockfish/*') if os.path.isfile(f) and os.access(f,os.X_OK)][0]
    shutil.copy(b,'tools/stockfish'); os.chmod('tools/stockfish',0o755)
import torch; print('torch',torch.__version__,'cuda',torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!echo 'sanity:'; echo -e 'uci\nquit' | ./tools/stockfish | grep '^id name'

In [ ]:
# persist data/ + bench/ to Drive (survive session restarts)
import os, shutil
for d in ('data','bench'):
    local=f'/content/chesshack/{d}'; target=f'{DRIVE}/{d}'
    if os.path.islink(local): continue
    if os.path.isdir(local): shutil.copytree(local,target,dirs_exist_ok=True); shutil.rmtree(local,ignore_errors=True)
    os.makedirs(target,exist_ok=True); os.symlink(target,local)
!ls -la data/distill | head

## Phase 1a — label positions with Stockfish (uses ALL cores)
~10 pos/s/core. On 48 cores ≈ 200+ pos/s → ~800k positions in ~1h. Resumable (re-run to append). Skip if shards already in Drive.

In [ ]:
import os
!python -m trainer.label --n 800000 --out data/distill --seed 1 --workers {os.cpu_count()}

## Phase 1b — distillation training (GPU, production net C256/B20)
Re-run to continue. Targets ~2000+ Elo.

In [ ]:
!python -m trainer.train --mode distill --data data/distill --net prod --steps 40000 --batch 1024 --lr 2e-3 --workers 16 --out data/nets/distilled.pt

In [ ]:
# headline Elo vs the Stockfish UCI_Elo ladder, + anti-stall sanity
!python -m bench.elo --ckpt data/nets/distilled.pt --games 40 --sims 800
!python -m bench.search_gain --ckpt data/nets/distilled.pt --sims 600 --games 40

## Phase 2 — self-play RL (uncapped, the ratchet)
Starts from the distilled net. CPU workers search, the GPU inference server coalesces leaves, the gate promotes only real gains. Watch `bench/elo_history.json` climb past 2900. Leaves a few cores for the trainer/server.

In [ ]:
import os
W=max(2, os.cpu_count()-4)
!python -m trainer.selfplay --init-from data/nets/distilled.pt --net prod --workers {W} --steps 300000 --batch 1024 --sims 600

## Notes
- **Resume:** re-run cells 1–6 in a new session; Drive has the shards/checkpoints. Distill/self-play continue.
- **Monitor Elo:** `import json; print(json.load(open('bench/elo_history.json'))[-1])`.
- **Bigger net later:** `--net` uses prod (24.5M); to scale to 45.5M, add a `scale` option / edit config.
- Free Colab disconnects ~12h; Drive persistence means no progress lost.
- 80GB lets you raise `--batch` and self-play `--workers`; watch GPU util + `avg_batch` in the logs.